# ml_D_svm_case.ipynb

**所属章节**：Chapter D 支持向量机  
**主题**：信用违约预测——比较 Logit、SVM（线性核）、SVM（RBF 核）、随机森林四种方法  
**数据**：模拟信贷数据（n=800，p=15个财务特征，二元违约标签）  
**运行说明**：顺序执行，约 2 分钟


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import (train_test_split, GridSearchCV,
                                      StratifiedKFold)
from sklearn.metrics import (roc_auc_score, classification_report,
                              RocCurveDisplay)
from sklearn.pipeline import Pipeline
import warnings, os
warnings.filterwarnings('ignore')
os.makedirs('figs', exist_ok=True)

FP  = fm.FontProperties(fname='/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc')
FPB = fm.FontProperties(fname='/usr/share/fonts/opentype/noto/NotoSansCJK-Medium.ttc')
C_color = {'primary':'#2166AC','secondary':'#D6604D','tertiary':'#4DAC26',
            'neutral':'#878787','highlight':'#B2182B','fill':'#D1E5F0'}
plt.rcParams.update({'figure.dpi':120,'savefig.dpi':300,
    'axes.spines.top':False,'axes.spines.right':False,
    'axes.grid':True,'grid.alpha':0.25,'grid.linestyle':'--'})
SEED=42; np.random.seed(SEED)
print('设置完成')

---
## 1. 数据生成（模拟信贷数据）

模拟 800 家企业的信贷特征与违约标签，含非线性风险因子交互效应。


In [ ]:
n, p = 800, 15
feat_names = ['资产负债率','流动比率','ROE','ROA','净利润增速',
              '营收增速','利息覆盖倍数','现金流比率','存货周转率',
              '应收账款周转率','总资产周转率','净资产收益率',
              '市净率','市盈率','股价波动率']

# 特征矩阵（模拟财务指标的相关结构）
rho = 0.3
Sig = rho**np.abs(np.subtract.outer(np.arange(p),np.arange(p)))
X_raw = np.random.multivariate_normal(np.zeros(p), Sig, n)

# 违约概率（含非线性交互：高负债 + 低流动性 → 高风险）
log_odds = (1.2*X_raw[:,0]         # 资产负债率（正向）
          - 0.9*X_raw[:,1]         # 流动比率（负向）
          - 0.7*X_raw[:,2]         # ROE（负向）
          + 0.5*X_raw[:,0]*X_raw[:,1]  # 交互项：高负债*低流动性
          + 0.3*X_raw[:,14]        # 波动率（正向）
          + np.random.normal(0,1,n))
prob = 1 / (1 + np.exp(-log_odds))
y = (prob > 0.5).astype(int)

print(f'样本量: {n}，特征数: {p}')
print(f'违约率: {y.mean():.1%}（{y.sum()} 家违约）')

# 分层抽样分割（保持类别比例）
X_tr_raw,X_te_raw,y_train,y_test = train_test_split(
    X_raw, y, test_size=0.25, stratify=y, random_state=SEED)
print(f'训练集: {X_tr_raw.shape[0]}，测试集: {X_te_raw.shape[0]}')

---
## 2. 四种方法的估计


In [ ]:
# 用 Pipeline 封装标准化（SVM 必须；RF/Logit 不必要但无害）
models = {
    'Logit': Pipeline([('sc',StandardScaler()),
                       ('mdl',LogisticRegression(C=1.0,random_state=SEED,max_iter=500))]),
    'SVM-线性': Pipeline([('sc',StandardScaler()),
                          ('mdl',SVC(kernel='linear',C=1.0,probability=True,random_state=SEED))]),
    'SVM-RBF': Pipeline([('sc',StandardScaler()),
                         ('mdl',SVC(kernel='rbf',C=10.0,gamma=0.1,probability=True,random_state=SEED))]),
    '随机森林': Pipeline([('sc',StandardScaler()),
                          ('mdl',RandomForestClassifier(n_estimators=200,random_state=SEED,n_jobs=-1))]),
}

results = {}
for name, pipe in models.items():
    pipe.fit(X_tr_raw, y_train)
    prob_te = pipe.predict_proba(X_te_raw)[:,1]
    auc = roc_auc_score(y_test, prob_te)
    acc = pipe.score(X_te_raw, y_test)
    results[name] = {'AUC':round(auc,4),'准确率':round(acc,4)}
    print(f'{name:<12} AUC={auc:.4f}，准确率={acc:.4f}')

---
## 3. SVM-RBF 超参数调优（C × gamma 网格搜索）


In [ ]:
# 网格搜索（C × gamma）
pipe_rbf = Pipeline([('sc',StandardScaler()),
                     ('svm',SVC(kernel='rbf',probability=True,random_state=SEED))])
param_grid = {
    'svm__C'    : [0.1, 1, 10, 100],
    'svm__gamma': [0.001, 0.01, 0.1, 1]
}
grid_cv = GridSearchCV(pipe_rbf, param_grid, cv=5,
                       scoring='roc_auc', n_jobs=-1)
grid_cv.fit(X_tr_raw, y_train)

print(f'最优参数：{grid_cv.best_params_}')
print(f'CV AUC = {grid_cv.best_score_:.4f}')

# 热力图：C × gamma 的 CV AUC
import pandas as pd
scores = grid_cv.cv_results_['mean_test_score'].reshape(4,4)
C_vals = [0.1,1,10,100]; g_vals = [0.001,0.01,0.1,1]

fig, ax = plt.subplots(figsize=(6,4.5))
im = ax.imshow(scores, cmap='Blues', aspect='auto', vmin=scores.min())
ax.set_xticks(range(4)); ax.set_yticks(range(4))
ax.set_xticklabels([str(g) for g in g_vals], fontproperties=FP)
ax.set_yticklabels([str(c) for c in C_vals], fontproperties=FP)
ax.set_xlabel('gamma', fontproperties=FP, fontsize=11)
ax.set_ylabel('C', fontproperties=FP, fontsize=11)
ax.set_title('SVM-RBF 超参数调优热力图（CV AUC）', fontproperties=FPB, fontsize=12)
for i in range(4):
    for j in range(4):
        ax.text(j,i,f'{scores[i,j]:.3f}', ha='center', va='center',
                fontsize=9, color='white' if scores[i,j]>0.7 else 'black')
plt.colorbar(im, ax=ax, label='CV AUC')
fig.tight_layout()
fig.savefig('figs/fig_D_case_heatmap.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()

---
## 4. ROC 曲线对比


In [ ]:
fig, ax = plt.subplots(figsize=(6, 5.5))
colors_roc = [C_color['tertiary'],C_color['neutral'],
               C_color['primary'],C_color['secondary']]

for (name, pipe), col in zip(models.items(), colors_roc):
    prob_te = pipe.predict_proba(X_te_raw)[:,1]
    RocCurveDisplay.from_predictions(
        y_test, prob_te, name=f'{name} (AUC={results[name]["AUC"]})',
        ax=ax, color=col, lw=2)

ax.plot([0,1],[0,1],'k--',lw=1,alpha=0.5,label='随机猜测')
ax.set_xlabel('假正例率（FPR）', fontproperties=FP, fontsize=11)
ax.set_ylabel('真正例率（TPR）', fontproperties=FP, fontsize=11)
ax.set_title('ROC 曲线对比（信用违约预测）', fontproperties=FPB, fontsize=13)
ax.legend(prop=FP, fontsize=8.5, loc='lower right')
fig.tight_layout()
fig.savefig('figs/fig_D_case_roc.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()

---
## 提示词模板 #6：SVM 分类

```
背景：用 SVM 做二分类（信用违约预测），需要调参并评估。

我的数据：X_train, y_train（y 为 0/1），X_test, y_test
特征数约 15，样本量约 600。

请帮我：
1. 用 Pipeline 打包 StandardScaler + SVC(kernel='rbf', probability=True)
2. GridSearchCV(cv=5, scoring='roc_auc')搜索 C:[0.01,0.1,1,10,100,1000]，gamma:[0.001,0.01,0.1,1,'scale']
3. 打印最优参数和 CV AUC
4. 计算测试集 AUC-ROC、精确率、召回率、F1
5. 绘制 ROC 曲线（与 Logit 对比）和 C×gamma 热力图
6. 所有代码中文注释，random_state=42
```


In [ ]:
# 汇总结果
print('='*50)
print('Chapter D 案例分析完成')
print('='*50)
df_r = pd.DataFrame(results).T
print(df_r.to_string())
best_m = df_r['AUC'].idxmax()
print(f'\n最优方法: {best_m}，AUC = {df_r.loc[best_m,"AUC"]:.4f}')
print('\n输出文件:')
print('  figs/fig_D_case_heatmap.png')
print('  figs/fig_D_case_roc.png')